## Library

In [17]:
import numpy as np
import pandas as pd
import time
import os

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

## Config

In [18]:

# ---------------- CONFIG ----------------
DATA_PATH = "../../data/full_data.xlsx"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

# TRAIN/TEST SPLIT
TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")  # training period end
TEST_START_DATE  = pd.Timestamp("2022-04-01")  # test period start

# === FILL THESE WITH YOUR BEST SARIMAX HYPERPARAMETERS FROM CV ===
# Example: (p,d,q) = (2,1,0), (P,D,Q,s) = (1,1,0,12)
BEST_ORDER          = (1, 0, 2)       # (p, d, q)
BEST_SEASONAL_ORDER = (0, 0, 0, 12)   # (P, D, Q, s)

# Exogenous features (same as other models)
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH"
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
]

exog_cols = continuous_cols + categorical_cols

## Metrics

In [19]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """Global MASE using in-sample seasonal naive with period m on y_train."""
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale

def directional_accuracy(df, entity_col, time_col, target_col, pred_col):
    """Fraction of times sign of month-on-month change is correct."""
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_diff"] = df.groupby(entity_col)[target_col].diff()
    df["yhat_diff"] = df.groupby(entity_col)[pred_col].diff()
    mask = df["y_diff"].notna() & df["yhat_diff"].notna()
    same_dir = np.sign(df.loc[mask, "y_diff"]) == np.sign(df.loc[mask, "yhat_diff"])
    return same_dir.mean()

def growth_rate_error(df, entity_col, time_col, target_col, pred_col, m=12):
    """
    12-month growth rate error:
    g_t = (y_t - y_{t-m}) / y_{t-m}
    Returns MAE of growth-rate error.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_lag_m"] = df.groupby(entity_col)[target_col].shift(m)
    df["yhat_lag_m"] = df.groupby(entity_col)[pred_col].shift(m)

    mask = df["y_lag_m"].notna() & df["yhat_lag_m"].notna() & (df["y_lag_m"] != 0)
    y_gr = (df.loc[mask, target_col] - df.loc[mask, "y_lag_m"]) / df.loc[mask, "y_lag_m"]
    yhat_gr = (df.loc[mask, pred_col] - df.loc[mask, "yhat_lag_m"]) / df.loc[mask, "yhat_lag_m"]

    return np.mean(np.abs(y_gr - yhat_gr))

def morans_i(
    residuals,
    xs,
    ys,
    k=5,
    eps=1e-8,
    symmetric=True,
    row_standardize=True,
    permutations=0,
    random_state=None,
):
    """
    Compute Moran's I for residuals using k-nearest neighbours
    with inverse-distance weights.
    """
    residuals = np.asarray(residuals, dtype=float)
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)

    N = len(residuals)
    if not (len(xs) == len(ys) == N):
        raise ValueError("residuals, xs, ys must all have the same length")

    # Center residuals
    x_mean = residuals.mean()
    x_dev = residuals - x_mean

    # Build kNN graph
    coords = np.column_stack([xs, ys])
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    # Weight matrix W (dense; for large N you might switch to sparse)
    W = np.zeros((N, N), dtype=float)
    for i in range(N):
        neigh_idx = indices[i, 1:]          # skip self at index 0
        w = 1.0 / (distances[i, 1:] + eps)  # inverse-distance weights
        W[i, neigh_idx] = w

    # Optional symmetrisation
    if symmetric:
        W = 0.5 * (W + W.T)

    # Optional row standardisation
    if row_standardize:
        row_sums = W.sum(axis=1, keepdims=True)
        W = np.where(row_sums > 0, W / (row_sums + eps), 0.0)

    S0 = W.sum()

    # Moran's I numerator and denominator (vectorised)
    num = (W * (x_dev[:, None] * x_dev[None, :])).sum()
    den = (x_dev ** 2).sum() + eps

    I_obs = (N / S0) * (num / den)

    result = {
        "I": I_obs,
        "S0": S0,
        "permutations": None,
        "z_score": None,
        "p_value": None,
    }

    # Optional permutation test
    if permutations > 0:
        if isinstance(random_state, np.random.Generator):
            rng = random_state
        else:
            rng = np.random.default_rng(random_state)

        perm_I = np.empty(permutations, dtype=float)
        for b in range(permutations):
            perm = rng.permutation(x_dev)
            num_b = (W * (perm[:, None] * perm[None, :])).sum()
            perm_I[b] = (N / S0) * (num_b / den)

        mean_perm = perm_I.mean()
        std_perm = perm_I.std(ddof=1) + eps
        z = (I_obs - mean_perm) / std_perm

        extreme = np.sum(np.abs(perm_I - mean_perm) >= np.abs(I_obs - mean_perm))
        p_val = (extreme + 1) / (permutations + 1)

        result.update(
            {
                "permutations": perm_I,
                "z_score": z,
                "p_value": p_val,
            }
        )

    return result

## Load data

In [20]:
df_full = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df_full = df_full.query('Date < "2024-03-31"')
df_full = df_full.sort_values([ENTITY_COL, TIME_COL]).reset_index(drop=True)

# robust centroids (for Moran's I)
df_full[["centroid_x", "centroid_y"]] = (
    df_full.groupby(ENTITY_COL)[["centroid_x", "centroid_y"]]
           .ffill()
           .bfill()
)

centroid_df = (
    df_full.drop_duplicates(ENTITY_COL)
           .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
)

# restrict to >= TRAIN_START_DATE
df = df_full[df_full[TIME_COL] >= TRAIN_START_DATE].copy()
df = df.sort_values([TIME_COL, ENTITY_COL]).reset_index(drop=True)

la_order = sorted(df[ENTITY_COL].unique())
N = len(la_order)
print("Number of LAs used:", N)

centroid_df = centroid_df.loc[la_order]
bad_las = centroid_df[centroid_df.isna().any(axis=1)].index.tolist()
if bad_las:
    print(f"⚠ Dropping {len(bad_las)} LAs with missing centroids:", bad_las)
    centroid_df = centroid_df.dropna()
    la_order = centroid_df.index.tolist()
    df = df[df[ENTITY_COL].isin(la_order)].copy()
    N = len(la_order)
    print("Updated number of LAs:", N)

# Ensure one row per (Date, LA)
df = (
    df.sort_values([TIME_COL, ENTITY_COL])
      .drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last")
      .copy()
)

dates = pd.Index(sorted(df[TIME_COL].unique()))
T_total = len(dates)
print("Total months in eval period:", T_total)

full_index = pd.MultiIndex.from_product(
    [dates, la_order],
    names=[TIME_COL, ENTITY_COL]
)

df_panel = (
    df.set_index([TIME_COL, ENTITY_COL])
      .reindex(full_index)
      .sort_index()
)

# ffill/bfill features + target within each LA
df_panel[exog_cols + [TARGET_COL]] = (
    df_panel[exog_cols + [TARGET_COL]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

missing_total = df_panel[exog_cols + [TARGET_COL]].isna().sum().sum()
if missing_total > 0:
    print(f"⚠ {missing_total} NaNs after ffill/bfill. Filling with column means.")
    col_means = df_panel[exog_cols + [TARGET_COL]].mean()
    df_panel[exog_cols + [TARGET_COL]] = df_panel[exog_cols + [TARGET_COL]].fillna(col_means)

print("NaNs after panel completion:",
      df_panel[exog_cols + [TARGET_COL]].isna().sum().sum())


Number of LAs used: 294
Total months in eval period: 204
NaNs after panel completion: 0


## Training

In [21]:
# =========================================================
# TRAIN / TEST SPLIT
# =========================================================
train_mask = (dates >= TRAIN_START_DATE) & (dates <= TRAIN_END_DATE)
test_mask  = (dates >= TEST_START_DATE)

train_dates = dates[train_mask]
test_dates  = dates[test_mask]

print(f"Train: {train_dates[0].date()} → {train_dates[-1].date()}")
print(f"Test : {test_dates[0].date()} → {test_dates[-1].date()}")

df_train = df_panel.loc[(train_dates, slice(None)), :].copy()
df_test  = df_panel.loc[(test_dates,  slice(None)), :].copy()

# =========================================================
# LEAK-FREE SCALING OF EXOGENOUS FEATURES
# =========================================================
scaler = StandardScaler()
scaler.fit(df_train[continuous_cols])

df_train_scaled = df_train.copy()
df_test_scaled  = df_test.copy()

df_train_scaled.loc[:, continuous_cols] = scaler.transform(df_train[continuous_cols])
df_test_scaled.loc[:,  continuous_cols] = scaler.transform(df_test[continuous_cols])

# For MASE scaling: all training targets in original units (all LAs)
y_train_all = df_train[TARGET_COL].values

# =========================================================
# FIT SARIMAX PER LA, FORECAST TEST PERIOD
# =========================================================
print("\n=== Fitting SARIMAX per LA and forecasting test period ===")
start_time = time.time()

rows_true = []
rows_pred = []
success_count = 0
fail_count = 0

for la in la_order:
    sub_train = df_train_scaled.xs(la, level=ENTITY_COL).sort_index()
    sub_test  = df_test_scaled.xs(la,  level=ENTITY_COL).sort_index()

    if sub_train[TARGET_COL].isna().all() or sub_test[TARGET_COL].isna().all():
        print(f"  LA {la}: missing train/test data; skipping.")
        fail_count += 1
        continue

    endog_train = sub_train[TARGET_COL].values.astype(float)
    exog_train  = sub_train[exog_cols].values.astype(float)
    exog_test   = sub_test[exog_cols].values.astype(float)
    n_test      = len(sub_test)

    if n_test == 0:
        print(f"  LA {la}: no test data; skipping.")
        fail_count += 1
        continue

    try:
        model = SARIMAX(
            endog=endog_train,
            exog=exog_train,
            order=BEST_ORDER,
            seasonal_order=BEST_SEASONAL_ORDER,
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        res = model.fit(disp=False)

        # forecast – this is usually a NumPy array, so DON'T use .values
        y_forecast = res.forecast(steps=n_test, exog=exog_test)
        y_pred = np.asarray(y_forecast, dtype=float)

        y_true = sub_test[TARGET_COL].values.astype(float)
        la_codes = np.array([la] * n_test)
        la_dates = sub_test.index.get_level_values(TIME_COL).to_numpy()

        df_true_la = pd.DataFrame({
            TIME_COL: la_dates,
            ENTITY_COL: la_codes,
            "y_true": y_true,
        })

        df_pred_la = pd.DataFrame({
            TIME_COL: la_dates,
            ENTITY_COL: la_codes,
            "y_pred": y_pred,
        })

    except Exception as e:
        print(f"  LA {la}: SARIMAX failed ({e}). Skipping this LA.")
        fail_count += 1
        continue

    # only reached if try block succeeds
    rows_true.append(df_true_la)
    rows_pred.append(df_pred_la)
    success_count += 1
    print(f"  LA {la}: fitted successfully, forecast {n_test} months.")

train_time = time.time() - start_time
print(f"\nTotal SARIMAX fitting + forecasting time: {train_time:.1f} seconds")
print(f"Successful LAs: {success_count} / {len(la_order)}, failed/skipped: {fail_count}")

# ---- safety check before concatenation ----
if not rows_true or not rows_pred:
    raise RuntimeError(
        f"No successful LA forecasts; cannot compute evaluation metrics "
        f"(rows_true={len(rows_true)}, rows_pred={len(rows_pred)})."
    )

df_true = pd.concat(rows_true, axis=0, ignore_index=True)
df_pred = pd.concat(rows_pred, axis=0, ignore_index=True)

df_test_eval = (
    df_true.merge(df_pred, on=[TIME_COL, ENTITY_COL], how="inner")
           .sort_values([TIME_COL, ENTITY_COL])
           .reset_index(drop=True)
)

df_test_eval["resid"] = df_test_eval["y_true"] - df_test_eval["y_pred"]

Train: 2007-04-01 → 2022-03-01
Test : 2022-04-01 → 2024-03-01

=== Fitting SARIMAX per LA and forecasting test period ===
  LA E06000001: fitted successfully, forecast 24 months.
  LA E06000002: fitted successfully, forecast 24 months.
  LA E06000003: fitted successfully, forecast 24 months.


C:\Users\slong\AppData\Local\Temp\ipykernel_25924\4268621518.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.0899061  -1.0899061  -1.0899061  ...  1.65712819  1.65712819
  4.40416248]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_train_scaled.loc[:, continuous_cols] = scaler.transform(df_train[continuous_cols])
C:\Users\slong\AppData\Local\Temp\ipykernel_25924\4268621518.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.0899061  -1.0899061   0.00890762 ...  1.65712819  1.65712819
  4.40416248]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_test_scaled.loc[:,  continuous_cols] = scaler.transform(df_test[continuous_cols])


  LA E06000004: fitted successfully, forecast 24 months.
  LA E06000005: fitted successfully, forecast 24 months.
  LA E06000006: fitted successfully, forecast 24 months.
  LA E06000007: fitted successfully, forecast 24 months.
  LA E06000008: fitted successfully, forecast 24 months.
  LA E06000009: fitted successfully, forecast 24 months.
  LA E06000010: fitted successfully, forecast 24 months.
  LA E06000011: fitted successfully, forecast 24 months.
  LA E06000012: fitted successfully, forecast 24 months.
  LA E06000013: fitted successfully, forecast 24 months.
  LA E06000014: fitted successfully, forecast 24 months.
  LA E06000015: fitted successfully, forecast 24 months.
  LA E06000016: fitted successfully, forecast 24 months.
  LA E06000017: fitted successfully, forecast 24 months.
  LA E06000018: fitted successfully, forecast 24 months.
  LA E06000019: fitted successfully, forecast 24 months.
  LA E06000020: fitted successfully, forecast 24 months.
  LA E06000021: fitted successf

## Evaluation

In [22]:
# =========================================================
# GLOBAL METRICS
# =========================================================
global_mae   = mae(df_test_eval["y_true"], df_test_eval["y_pred"])
global_rmse  = rmse(df_test_eval["y_true"], df_test_eval["y_pred"])
global_smape = smape(df_test_eval["y_true"], df_test_eval["y_pred"])
global_mase  = mase(df_test_eval["y_true"], df_test_eval["y_pred"], y_train_all, m=12)

print("\n=== Global accuracy ===")
print(f"MAE   : {global_mae:,.3f}")
print(f"RMSE  : {global_rmse:,.3f}")
print(f"sMAPE : {global_smape:.3f}%")
print(f"MASE  : {global_mase:.3f}")

# =========================================================
# ACROSS-LA CONSISTENCY
# =========================================================
la_mae = (
    df_test_eval.groupby(ENTITY_COL)
                .apply(lambda g: mae(g["y_true"], g["y_pred"]))
)

median_mae = float(np.median(la_mae.values))
p75_mae    = float(np.percentile(la_mae.values, 75))

print("\n=== Across-LA consistency ===")
print(f"Median LA MAE       : {median_mae:,.3f}")
print(f"75th percentile MAE : {p75_mae:,.3f}")

# =========================================================
# SPATIO-TEMPORAL DIAGNOSTICS
# =========================================================
# Moran's I on mean residual per LA
la_resid_mean = df_test_eval.groupby(ENTITY_COL)["resid"].mean()
centroids = centroid_df.loc[la_resid_mean.index][["centroid_x", "centroid_y"]]

common = la_resid_mean.index.intersection(centroids.index)
la_resid_mean = la_resid_mean.loc[common]
centroids = centroids.loc[common]

mask_valid = centroids[["centroid_x", "centroid_y"]].notna().all(axis=1)
centroids_valid = centroids.loc[mask_valid]
la_resid_mean_valid = la_resid_mean.loc[centroids_valid.index]

print(f"\nLAs used for Moran's I: {len(centroids_valid)} / {len(la_resid_mean)}")

if len(centroids_valid) <= 1:
    I_moran = {"I": np.nan, "z_score": np.nan, "p_value": np.nan}
else:
    k_eff = min(5, len(centroids_valid) - 1)
    I_moran = morans_i(
        residuals=la_resid_mean_valid.values,
        xs=centroids_valid["centroid_x"].values,
        ys=centroids_valid["centroid_y"].values,
        k=k_eff,
        permutations=999,
        random_state=42,
    )
    print("\n=== Spatio-temporal diagnostics ===")
    print(f"Moran's I (mean residuals across LAs): {I_moran['I']:.4f}")
    print(f"Moran's I z score: {I_moran['z_score']:.4f}")
    print(f"Moran's I p value: {I_moran['p_value']:.4f}")

# Ljung–Box on mean residual over time
monthly_resid = (
    df_test_eval.groupby(TIME_COL)["resid"]
                .mean()
                .sort_index()
)

lb = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)
lb_stat = float(lb["lb_stat"].iloc[0])
lb_p    = float(lb["lb_pvalue"].iloc[0])

print("\n=== Spatio-temporal diagnostics ===")
print(f"Moran's I (mean residuals across LAs): {I_moran['I']:.4f}")
print(f"Moran's I z score: {I_moran['z_score']:.4f}")
print(f"Moran's I p value: {I_moran['p_value']:.4f}")
print(f"Ljung–Box Q(12): stat={lb_stat:.3f}, p={lb_p:.4f}")

# =========================================================
# DIRECTIONAL ACCURACY & GROWTH-RATE ERROR
# =========================================================
dir_acc = directional_accuracy(df_test_eval, ENTITY_COL, TIME_COL, "y_true", "y_pred")

# approximate 12-month growth-rate error
# build full [T, N] arrays for y_true & y_pred on test-support
dates_all = dates
la_array  = np.array(la_order)

# map (Date, LA) in df_test_eval back to indices
date_to_idx = {d: i for i, d in enumerate(dates_all)}

y_true_full = np.full((T_total, N), np.nan, dtype=float)
y_pred_full = np.full((T_total, N), np.nan, dtype=float)

for _, row in df_test_eval.iterrows():
    t = date_to_idx[row[TIME_COL]]
    n = np.where(la_array == row[ENTITY_COL])[0][0]
    y_true_full[t, n] = row["y_true"]
    y_pred_full[t, n] = row["y_pred"]

errs = []
test_start_idx = np.where(dates_all == TEST_START_DATE)[0][0]

for t in range(test_start_idx, T_total):
    t_prev = t - 12
    if t_prev < 0:
        continue
    true_t    = y_true_full[t]      # [N]
    true_prev = y_true_full[t_prev] # [N]
    pred_t    = y_pred_full[t]      # [N]
    mask = (~np.isnan(pred_t)) & (~np.isnan(true_t)) & (~np.isnan(true_prev)) & \
           (true_t > 0) & (true_prev > 0)
    if not mask.any():
        continue

    true_growth = np.log(true_t[mask]) - np.log(true_prev[mask])
    pred_growth = np.log(pred_t[mask]) - np.log(true_prev[mask])
    errs.append(np.abs(true_growth - pred_growth))

if errs:
    gre_mae = float(np.mean(np.concatenate(errs)))
else:
    gre_mae = np.nan

print("\n=== Direction & growth ===")
print(f"Directional accuracy (MoM sign)   : {dir_acc:.3f}")
print(f"Growth-rate error MAE (12-month)  : {gre_mae:.4f}")


=== Global accuracy ===
MAE   : 12,270.508
RMSE  : 27,477.521
sMAPE : 3.651%
MASE  : 0.120

=== Across-LA consistency ===
Median LA MAE       : 7,621.596
75th percentile MAE : 13,548.815

LAs used for Moran's I: 294 / 294

=== Spatio-temporal diagnostics ===
Moran's I (mean residuals across LAs): -0.0255
Moran's I z score: -0.6592
Moran's I p value: 0.4730

=== Spatio-temporal diagnostics ===
Moran's I (mean residuals across LAs): -0.0255
Moran's I z score: -0.6592
Moran's I p value: 0.4730
Ljung–Box Q(12): stat=31.257, p=0.0018


C:\Users\slong\AppData\Local\Temp\ipykernel_25924\3206532046.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mae(g["y_true"], g["y_pred"]))



=== Direction & growth ===
Directional accuracy (MoM sign)   : 0.618
Growth-rate error MAE (12-month)  : 0.0402


## Results

In [15]:

# =========================================================
# SAVE SUMMARY METRICS TO CSV
# =========================================================
output_path = "../../results/sarimax_final_test_results.xlsx"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

summary_df = pd.DataFrame([{
    "model_type": "SARIMAX_panel",
    "order": str(BEST_ORDER),
    "seasonal_order": str(BEST_SEASONAL_ORDER),

    # global metrics
    "MAE": global_mae,
    "RMSE": global_rmse,
    "sMAPE": global_smape,
    "MASE": global_mase,

    # across-LA
    "Median_LA_MAE": median_mae,
    "P75_LA_MAE": p75_mae,

    # spatio-temporal
    "Morans_I": float(I_moran["I"]) if isinstance(I_moran, dict) else np.nan,
    "Morans_I_z": float(I_moran["z_score"]) if isinstance(I_moran, dict) and I_moran.get("z_score") is not None else np.nan,
    "Morans_I_p": float(I_moran["p_value"]) if isinstance(I_moran, dict) and I_moran.get("p_value") is not None else np.nan,
    "LjungBox_Q12": lb_stat,
    "LjungBox_p": lb_p,

    # directional & growth
    "Directional_Accuracy": dir_acc,
    "GrowthRateError_MAE": gre_mae,

    # efficiency
    "Training_Time_sec": train_time,
    "Train_End_Date": TRAIN_END_DATE.strftime("%Y-%m-%d"),
    "Test_Start_Date": TEST_START_DATE.strftime("%Y-%m-%d"),
}])

la_mae_df = la_mae.reset_index()
la_mae_df.columns = [ENTITY_COL, "LA_MAE"]

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    summary_df.to_excel(writer, sheet_name="Global_Summary", index=False)
    la_mae_df.to_excel(writer, sheet_name="LA_MAE", index=False)
    df_test_eval[[ENTITY_COL, TIME_COL, "y_true", "y_pred", "resid"]].to_excel(
        writer, sheet_name="Test_Predictions", index=False
    )

print(f"\nResults saved to: {output_path}")
print("\n✅ Final SARIMAX evaluation metrics saved to:")
print(output_path)


PermissionError: [Errno 13] Permission denied: '../../results/sarimax_final_test_results.xlsx'

In [ ]:
df_test_eval

,Date,AreaCode,y_true,y_pred,resid
0,2022-04-01,E06000001,120831.0,1.172638e+05,3567.207196
1,2022-04-01,E06000002,121586.0,1.294676e+05,-7881.618747
2,2022-04-01,E06000003,137782.0,1.390759e+05,-1293.937319
3,2022-04-01,E06000004,149654.0,1.477044e+05,1949.582114
4,2022-04-01,E06000005,140339.0,1.406634e+05,-324.434297
...,...,...,...,...,...
7051,2024-03-01,E09000029,432144.0,4.132268e+05,18917.249557
7052,2024-03-01,E09000030,488046.0,5.474930e+05,-59447.039222
7053,2024-03-01,E09000031,499866.0,4.742189e+05,25647.141657
7054,2024-03-01,E09000032,721959.0,6.482153e+05,73743.692489
